In [ ]:
!pip install -q data-designer openai

In [ ]:
!pip freeze | grep -E 'data-designer|openai' > requirements_synth_data_gen.txt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_BARTA')


In [ ]:
from data_designer.essentials import (
    ChatCompletionInferenceParams,
    DataDesigner,
    DataDesignerConfigBuilder,
    ModelConfig,
)

In [ ]:
data_designer = DataDesigner()

In [ ]:
MODEL_PROVIDER = "openai"
MODEL_ID = "gpt-4.1"
MODEL_ALIAS = "openai-text"

model_configs = [
 ModelConfig(alias=MODEL_ALIAS, model=MODEL_ID,
             inference_parameters=ChatCompletionInferenceParams(
                 generation_type='chat-completion',
                 max_parallel_requests=4,
                 timeout=None,
                 extra_body=None,
                 temperature=0.8,
                 top_p=None,
                 max_tokens=None), provider=MODEL_PROVIDER),
]

In [ ]:
config_builder = DataDesignerConfigBuilder(model_configs=model_configs)
print(config_builder.model_configs)

[ModelConfig(alias='openai-text', model='gpt-4.1', inference_parameters=ChatCompletionInferenceParams(generation_type='chat-completion', max_parallel_requests=4, timeout=None, extra_body=None, temperature=0.8, top_p=None, max_tokens=None), provider='openai')]


In [ ]:
local_filename = "/content/t2g.csv" # change it to t2g CSV

# Seed datasets are passed as reference objects to the config builder.
seed_dataset_reference = data_designer.make_seed_reference_from_file(local_filename)

config_builder.with_seed_dataset(seed_dataset_reference)

DataDesignerConfigBuilder(
    seed_dataset: '/content/t2g.csv'
    seed_dataset_columns: ['Text', 'Gloss', 'Topics']
)

In [ ]:
PROMPT = """\
You are a linguistically careful assistant trained to generate Bangla Sign Language (BdSL) gloss approximations from Bangla sentences.
You are given a seed dataset with two columns:

{{Text}} translates to {{Gloss}}

Your task is to learn the mapping pattern between Sentence → Gloss from the examples and then generate
ONE new, high-quality gloss for ONE  Bangla sentence for the topic "{{Topics}}".

Strict instructions:
1. Use Bangla words, written in Bangla script.
2. Use the correct infinitive form of verbs wherever applicable
Example:
“কাজ করে” → “কাজ করা”
“গাইতে” → “গাওয়া”
Do not invent verb forms not seen or implied in the seed data.
3. Follow BdSL-preferred ordering as observed in the seed Gloss column.
4. Do not paraphrase or add information.
5. Do not remove core meaning.
6. If a sentence is ambiguous, choose the most neutral, literal gloss.
7. No numbering, no quotes, no extra whitespace.
8. If the input sentence contains:
- multiple clauses
- temporal expressions
- conditionals
- negation
- modality
- questions
The gloss MUST preserve that complexity.
9. Final output MUST strictly follow this format:

BanglaSentence -> BanglaGloss

Constraints:
- Exactly ONE line.
- Exactly ONE "->".
- Left side MUST be the input sentence verbatim.
- Right side MUST be the generated BdSL gloss.
- A simpler gloss than the input sentence is INVALID.
- Any deviation is an error.
"""

In [ ]:
config_builder.add_column(
    name="gloss_form",
    column_type="llm-text",
    prompt=PROMPT,
    model_alias=MODEL_ALIAS,
)

DataDesignerConfigBuilder(
    seed_dataset: '/content/t2g.csv'
    seed_dataset_columns: ['Text', 'Gloss', 'Topics']
    llm_text_columns: ['gloss_form']
)

In [ ]:
config_builder.validate()

[07:13:16] [INFO] ✅ Validation passed


DataDesignerConfigBuilder(
    seed_dataset: '/content/t2g.csv'
    seed_dataset_columns: ['Text', 'Gloss', 'Topics']
    llm_text_columns: ['gloss_form']
)

In [ ]:
preview = data_designer.preview(config_builder, num_records=5)

[07:13:16] [INFO] 🕵️ Preview generation in progress
[07:13:16] [INFO] ✅ Validation passed
[07:13:16] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[07:13:16] [INFO] 🩺 Running health checks for models...
[07:13:16] [INFO]   |-- 👀 Checking 'gpt-4.1' in provider named 'openai' for model alias 'openai-text'...
[07:13:17] [INFO]   |-- ✅ Passed!
[07:13:17] [INFO] 🌱 Sampling 5 records from seed dataset
[07:13:17] [INFO]   |-- seed dataset size: 1000 records
[07:13:17] [INFO]   |-- sampling strategy: ordered
[07:13:17] [INFO] Preparing llm-text column generation
[07:13:17] [INFO]   |-- column name: 'gloss_form'
[07:13:17] [INFO]   |-- model config:
{
    "alias": "openai-text",
    "model": "gpt-4.1",
    "inference_parameters": {
        "generation_type": "chat-completion",
        "max_parallel_requests": 4,
        "timeout": null,
        "extra_body": null,
        "temperature": 0.8,
        "top_p": null,
        "max_tokens": null
    },
    "provider": "openai"
}
[07

In [ ]:
# Run this cell multiple times to cycle through the 2 preview records.
preview.display_sample_record()

                                                   Seed Columns                                                    
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name                       ┃ Value                                                                              ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Text                       │ আমি প্রতিদিন সকালে ঘুম থেকে উঠি।                                                             │
├────────────────────────────┼────────────────────────────────────────────────────────────────────────────────────┤
│ Gloss                      │ আমি প্রতিদিন সকাল ঘুম থেকে উঠা                                                              │
├────────────────────────────┼────────────────────────────────────────────────────────────────────────────────────┤
│ Topics                     │ Transportation                                                                     │
└────────────────────────────┴────────────────────────────────────────────────────────────────────────────────────┘
                                                                                                                   
                                                                                                                   
                                                 Generated Columns                                                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name                          ┃ Value                                                                           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gloss_form                    │ আমি বাসে স্কুলে যাই। -> আমি বাস স্কুল যাওয়া                                                  │
└───────────────────────────────┴─────────────────────────────────────────────────────────────────────────────────┘
                                                                                                                   
                                                    [index: 0]

In [ ]:
# The preview dataset is available as a pandas DataFrame.
preview.dataset

,Text,Gloss,Topics,gloss_form
0,আমি প্রতিদিন সকালে ঘুম থেকে উঠি।,আমি প্রতিদিন সকাল ঘুম থেকে উঠা,Transportation,আমি বাসে স্কুলে যাই। -> আমি বাস স্কুল যাওয়া
1,মা আজ খুব ভালো ইলিশ মাছ রান্না করেছেন।,মা আজ খুব ভালো ইলিশ মাছ রান্না করা,News discussions,আজকের খবর নিয়ে সবাই আলোচনা করছে। -> আজ খবর সবা...
2,আমার ভাই বোনদের সাথে খেলতে ভালোবাসে।,আমার ভাই বোন একসাথে খেলা ভালোবাসা,Learning new skills,নতুন দক্ষতা শিখতে আমি আগ্রহী। -> আমি নতুন দক্ষ...
3,তুমি কি দুপুরের খাবার খেয়েছো?,তুমি দুপুরের খাবার খাওয়া,Exams and tests,তোমার পরীক্ষা কবে শুরু হবে? -> তোমার পরীক্ষা ক...
4,বাবা সন্ধ্যায় বাজার থেকে ফিরবেন।,বাবা সন্ধ্যা বাজার ফিরা,Holiday traditions,ঈদের দিন সকলে নতুন কাপড় পরে। -> ঈদ দিন সবাই ন...


In [ ]:
# Print the analysis as a table.
preview.analysis.to_report()

──────────────────────────────────────── 🎨 Data Designer Dataset Profile ─────────────────────────────────────────

                                                                                                                   
                                                 Dataset Overview                                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ number of records               ┃ number of columns               ┃ percent complete records                    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 5                               │ 4                               │ 100.0%                                      │
└─────────────────────────────────┴─────────────────────────────────┴─────────────────────────────────────────────┘
                                                                                                                   
                                                                                                                   
                                              🌱 Seed-Dataset Columns                                              
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ column name                    ┃                 data type ┃                               number unique values ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Text                           │                    string │                                         5 (100.0%) │
├────────────────────────────────┼───────────────────────────┼────────────────────────────────────────────────────┤
│ Gloss                          │                    string │                                         5 (100.0%) │
├────────────────────────────────┼───────────────────────────┼────────────────────────────────────────────────────┤
│ Topics                         │                    string │                                         5 (100.0%) │
└────────────────────────────────┴───────────────────────────┴────────────────────────────────────────────────────┘
                                                                                                                   
                                                                                                                   
                                                📝 LLM-Text Columns                                                
┏━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                  ┃               ┃                              ┃       prompt tokens ┃       completion tokens ┃
┃ column name      ┃     data type ┃         number unique values ┃          per record ┃              per record ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gloss_form       │        string │                   5 (100.0%) │       414.0 +/- 9.8 │           73.0 +/- 11.5 │
└──────────────────┴───────────────┴──────────────────────────────┴─────────────────────┴─────────────────────────┘
                                                                                                                   
                                                                                                                   
╭────────────────────────────────────────────────── Table Notes ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  1. All token statistics are based on a sample of max(1000, len(dataset)) records.                              │
│  2. Tokens are calculated using tiktoken's cl100k_base t

In [ ]:
results = data_designer.create(config_builder, num_records=4000, dataset_name="syntheticGloss1000")

[07:17:12] [INFO] 🎨 Creating Data Designer dataset
[07:17:12] [INFO] ✅ Validation passed
[07:17:12] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[07:17:12] [INFO] 📂 Dataset path '/content/artifacts/syntheticGloss1000' already exists. Dataset from this session
		     will be saved to '/content/artifacts/syntheticGloss1000_01-01-2026_071712' instead.
[07:17:12] [INFO] 🩺 Running health checks for models...
[07:17:12] [INFO]   |-- 👀 Checking 'gpt-4.1' in provider named 'openai' for model alias 'openai-text'...
[07:17:12] [INFO]   |-- ✅ Passed!
[07:17:12] [INFO] ⏳ Processing batch 1 of 4
[07:17:13] [INFO] 🌱 Sampling 1000 records from seed dataset
[07:17:13] [INFO]   |-- seed dataset size: 1000 records
[07:17:13] [INFO]   |-- sampling strategy: ordered
[07:17:13] [INFO] Preparing llm-text column generation
[07:17:13] [INFO]   |-- column name: 'gloss_form'
[07:17:13] [INFO]   |-- model config:
{
    "alias": "openai-text",
    "model": "gpt-4.1",
    "inference_parameters": 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[07:19:51] [INFO] Preparing llm-text column generation
[07:19:51] [INFO]   |-- column name: 'gloss_form'
[07:19:51] [INFO]   |-- model config:
{
    "alias": "openai-text",
    "model": "gpt-4.1",
    "inference_parameters": {
        "generation_type": "chat-completion",
        "max_parallel_requests": 4,
        "timeout": null,
        "extra_body": null,
        "temperature": 0.8,
        "top_p": null,
        "max_tokens": null
    },
    "provider": "openai"
}
[07:19:51] [INFO] 🐙 Processing llm-text column 'gloss_form' with 4 concurrent workers
[07:22:27] [INFO] ⏳ Processing batch 3 of 4
[07:22:27] [INFO] 🌱 Sampling 1000 records from seed dataset
[07:22:27] [INFO]   |-- seed dataset size: 1000 records
[07:22:27] [INFO]   |-- sampling strategy: ordered


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[07:22:27] [INFO] Preparing llm-text column generation
[07:22:27] [INFO]   |-- column name: 'gloss_form'
[07:22:27] [INFO]   |-- model config:
{
    "alias": "openai-text",
    "model": "gpt-4.1",
    "inference_parameters": {
        "generation_type": "chat-completion",
        "max_parallel_requests": 4,
        "timeout": null,
        "extra_body": null,
        "temperature": 0.8,
        "top_p": null,
        "max_tokens": null
    },
    "provider": "openai"
}
[07:22:27] [INFO] 🐙 Processing llm-text column 'gloss_form' with 4 concurrent workers
[07:25:10] [INFO] ⏳ Processing batch 4 of 4
[07:25:10] [INFO] 🌱 Sampling 1000 records from seed dataset
[07:25:10] [INFO]   |-- seed dataset size: 1000 records
[07:25:10] [INFO]   |-- sampling strategy: ordered


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[07:25:10] [INFO] Preparing llm-text column generation
[07:25:10] [INFO]   |-- column name: 'gloss_form'
[07:25:10] [INFO]   |-- model config:
{
    "alias": "openai-text",
    "model": "gpt-4.1",
    "inference_parameters": {
        "generation_type": "chat-completion",
        "max_parallel_requests": 4,
        "timeout": null,
        "extra_body": null,
        "temperature": 0.8,
        "top_p": null,
        "max_tokens": null
    },
    "provider": "openai"
}
[07:25:10] [INFO] 🐙 Processing llm-text column 'gloss_form' with 4 concurrent workers
[07:27:49] [INFO] 📊 Model usage summary:
{
    "gpt-4.1": {
        "token_usage": {
            "input_tokens": 1373740,
            "output_tokens": 86116,
            "total_tokens": 1459856
        },
        "request_usage": {
            "successful_requests": 4000,
            "failed_requests": 0,
            "total_requests": 4000
        },
        "tokens_per_second": 2293,
        "requests_per_minute": 377
    }
}
[07:27:49

In [ ]:
# Load the generated dataset as a pandas DataFrame.
dataset = results.load_dataset()

dataset

,Text,Gloss,Topics,gloss_form
0,আমি প্রতিদিন সকালে ঘুম থেকে উঠি।,আমি প্রতিদিন সকাল ঘুম থেকে উঠা,Transportation,আমি বাসে স্কুলে যাই। -> আমি বাস স্কুল যাওয়া
1,মা আজ খুব ভালো ইলিশ মাছ রান্না করেছেন।,মা আজ খুব ভালো ইলিশ মাছ রান্না করা,News discussions,আজকের খবর নিয়ে আলোচনা হচ্ছে। -> আজ খবর আলোচনা...
2,আমার ভাই বোনদের সাথে খেলতে ভালোবাসে।,আমার ভাই বোন একসাথে খেলা ভালোবাসা,Learning new skills,নতুন দক্ষতা শেখা গুরুত্বপূর্ণ। -> নতুন দক্ষতা ...
3,তুমি কি দুপুরের খাবার খেয়েছো?,তুমি দুপুরের খাবার খাওয়া,Exams and tests,তোমার পরীক্ষা কখন শুরু হবে? -> তোমার পরীক্ষা ক...
4,বাবা সন্ধ্যায় বাজার থেকে ফিরবেন।,বাবা সন্ধ্যা বাজার ফিরা,Holiday traditions,পরিবারের সবাই একসাথে ঈদ পালন করে। -> পরিবার সব...
...,...,...,...,...
3995,"তুমি যদি মনোযোগ দাও, তাহলে আরও ভালো করতে পারবে।","তুমি মনোযোগ দেওয়া, আরও ভালো পারা।",Teachers and classmates,শিক্ষকরা আমাদের অনেক কিছু শেখান এবং সহপাঠীরা এ...
3996,আমি চাই আগামী বছর আরও বেশি ভ্রমণ করতে।,আমি চাই আগামী বছর অনেক ভ্রমণ করা,Cultural traditions,বাংলাদেশে নববর্ষ উপলক্ষে পহেলা বৈশাখ উদযাপন কর...
3997,বহু বিতর্কের পর অবশেষে প্রস্তাবিত অবকাঠামো উন্...,"অনেক তর্ক, শেষে নতুন অবকাঠামো নিয়ম সংসদ পাস শেষ।",Work and office,আমি আজ অফিসে অনেক কাজ করেছি -> আজ অফিস অনেক কা...
3998,আন্তর্জাতিক বাজারে জ্বালানির মূল্য বৃদ্ধি পাওয়...,"আন্তর্জাতিক বাজার তেল দাম বেশি, সরকার জরুরি আম...",Photography,ছবি তোলার জন্য ভালো ক্যামেরা দরকার। -> ছবি তুল...


In [ ]:
dataset.to_csv("syntheticGloss1000.csv", index=False)

In [ ]:
# Load the analysis results into memory.
analysis = results.load_analysis()

analysis.to_report()

──────────────────────────────────────── 🎨 Data Designer Dataset Profile ─────────────────────────────────────────

                                                                                                                   
                                                 Dataset Overview                                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ number of records               ┃ number of columns               ┃ percent complete records                    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 4,000                           │ 4                               │ 100.0%                                      │
└─────────────────────────────────┴─────────────────────────────────┴─────────────────────────────────────────────┘
                                                                                                                   
                                                                                                                   
                                              🌱 Seed-Dataset Columns                                              
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ column name                    ┃                 data type ┃                               number unique values ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Text                           │                    string │                                        998 (24.9%) │
├────────────────────────────────┼───────────────────────────┼────────────────────────────────────────────────────┤
│ Gloss                          │                    string │                                        989 (24.7%) │
├────────────────────────────────┼───────────────────────────┼────────────────────────────────────────────────────┤
│ Topics                         │                    string │                                          64 (1.6%) │
└────────────────────────────────┴───────────────────────────┴────────────────────────────────────────────────────┘
                                                                                                                   
                                                                                                                   
                                                📝 LLM-Text Columns                                                
┏━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                  ┃               ┃                              ┃        prompt tokens ┃      completion tokens ┃
┃ column name      ┃     data type ┃         number unique values ┃           per record ┃             per record ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gloss_form       │        string │                 3287 (82.2%) │       417.0 +/- 22.5 │          77.0 +/- 22.3 │
└──────────────────┴───────────────┴──────────────────────────────┴──────────────────────┴────────────────────────┘
                                                                                                                   
                                                                                                                   
╭────────────────────────────────────────────────── Table Notes ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  1. All token statistics are based on a sample of max(1000, len(dataset)) records.                              │
│  2. Tokens are calculated using tiktoken's cl100k_base t